In [ ]:
import os, sys, json, hashlib, shutil, zipfile, base64, zlib, glob, inspect
import numpy as np
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))

for p in glob.glob('/kaggle/input/**/solution_io.py', recursive=True):
    sp = os.path.dirname(os.path.dirname(os.path.dirname(p)))
    if sp not in sys.path:
        sys.path.insert(0, sp)
    break

from invsqrt import verify_submission_invsqrt as IV
from dqi import verify_submission_dqi as DV
from dfthcsos.verify import verify as CV, load_inputs, evaluate_words, passes_accuracy

print('load_inputs signature:', inspect.signature(load_inputs))
print('Environment ready')


In [ ]:
import shutil
bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))
shutil.copy2(bp_members / 'dfthcsos__submission.npz', sub_dir / 'dfthcsos' / 'submission.npz')

fac = dict(np.load(sub_dir / 'dfthcsos' / 'submission.npz', allow_pickle=False))
print('Factorization keys:', list(fac.keys()))
for k in fac:
    print(f'  {k}: shape={fac[k].shape}, dtype={fac[k].dtype}')

W = np.array(fac['W'], dtype=np.float64)
H2_I = np.array(fac['H2_I'], dtype=np.float64)
print('W shape:', W.shape, 'H2_I shape:', H2_I.shape)


In [ ]:
from scipy.stats import ortho_group

def apply_gauge(W, H2_I, O):
    R, B, D = W.shape
    A = np.concatenate([W, H2_I[:, None, :]], axis=1)
    A1 = A @ O
    W_new = A1[:, :-1, :]
    H2_I_new = A1[:, -1, :]
    offset_new = H2_I_new - W_new.sum(axis=1)
    return W_new, H2_I_new, offset_new

best_cost = float('inf')
best_O = None
D = W.shape[2]

for trial in range(50):
    O_trial = ortho_group.rvs(D)
    W_new, H2_I_new, offset_new = apply_gauge(W, H2_I, O_trial)
    A = np.concatenate([W, H2_I[:, None, :]], axis=1)
    A1 = np.concatenate([W_new, H2_I_new[:, None, :]], axis=1)
    gram_diff = np.max(np.abs(A @ A.swapaxes(1,2) - A1 @ A1.swapaxes(1,2)))
    if gram_diff > 1e-8:
        continue
    cost_proxy = np.sum(np.abs(W_new)) + np.sum(np.abs(H2_I_new))
    if cost_proxy < best_cost:
        best_cost = cost_proxy
        best_O = O_trial.copy()
    if trial % 10 == 0:
        print(f'Trial {trial}: best_cost={best_cost:.2f}, gram_diff={gram_diff:.2e}')

print(f'Best cost_proxy: {best_cost:.2f}')
np.save(WORK / 'best_gauge_46.npy', best_O)


In [ ]:
best_O = np.load(WORK / 'best_gauge_46.npy')
W_new, H2_I_new, offset_new = apply_gauge(W, H2_I, best_O)

fac_new = dict(fac)
fac_new['W'] = W_new.astype(fac['W'].dtype)
fac_new['H2_I'] = H2_I_new.astype(fac['H2_I'].dtype)
if 'offset_W' in fac_new:
    fac_new['offset_W'] = offset_new.astype(fac['offset_W'].dtype)

np.savez(sub_dir / 'dfthcsos' / 'submission.npz', **fac_new)

# Try to verify accuracy
try:
    from starter.task1_factorization.solution_io import read_factorization
    params, _ = read_factorization(sub_dir / 'dfthcsos' / 'submission.npz')
    inputs = load_inputs('FeMoCo76o113e', params)
    tree_edges = np.array(fac['tree_edges'])
    print('Loaded inputs successfully')
except Exception as e:
    print('Could not load inputs:', e)
    inputs = None

if inputs is not None:
    try:
        from starter.task2_angles.encode import encode_inputs
        words_new = encode_inputs(inputs, tree_edges, 8, 'nearest')
        metrics = evaluate_words(inputs, words_new, tree_edges, 8)
        print('Metrics:', json.dumps({k: (float(v) if not isinstance(v, bool) else v) for k, v in metrics.items()}, indent=2))
        print('Passes:', passes_accuracy(metrics))
    except Exception as e:
        print('Could not verify angle words:', e)
else:
    print('Skipping angle word verification')


In [ ]:
# Full verify
r = CV(sub_dir / 'dfthcsos')
print('Full verify:', r['status'], r['score'])

# Package
zip_path = WORK / 'submission_gauge.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(sub_dir.rglob('*')):
        if p.is_file():
            z.write(p, str(p.relative_to(sub_dir)))
print('Wrote', zip_path, zip_path.stat().st_size, 'bytes')
